In [1]:
import sys
from pathlib import Path

import torch
from torchvision import transforms

torch.backends.cuda.matmul.allow_tf32 = False
torch.backends.cudnn.allow_tf32 = False

this_path = Path(__file__) if '__file__' in globals() else Path("<unknown>.ipynb").resolve()
work_path = next((p for p in this_path.parents if p.name == "research"), None)
tools_path = work_path / Path("../torch-tools")
sys.path.append(str(tools_path))

from datasets import Datasets
from run_manager import RunManager, RunsManager
from trainer import Trainer, MultiTrainer, MergeEnsemble
from modules import CrossEntropyLossT, FixedLoss
from hook import HookManager
import utils

from models.resnet_ee import resnet18 as offnet
from models.resnet_git_ee_def import resnet18 as gitnet_def
from models.resnet_git_ee import resnet18 as gitnet
from models.resnet_git_ee_bn import resnet18 as gitnet_bn
from models.resnet_git_ee_test import resnet18 as gitnet_test

fetch_ds = Datasets(root=work_path / "assets/datasets/")

base_train_ds = fetch_ds("cifar10_train")
base_val_ds = fetch_ds("cifar10_val")

utils.torch_fix_seed(42)
train_trans = [transforms.RandomCrop(32, padding=4), transforms.RandomHorizontalFlip(), transforms.RandomRotation(15), transforms.ToTensor(), base_train_ds.normalizer()]
val_trans = [transforms.ToTensor(), base_train_ds.normalizer()]

train_ds = base_train_ds.transform(train_trans).in_ndata(1024)
val_ds = base_val_ds.transform(val_trans).in_ndata(1024)

train_dl = train_ds.loader(batch_size=128)
val_dl = val_ds.loader(batch_size=128)

epochs = 5
max_lr = 0.1
# max_lr = 0.005
# exp_name = "exp_compare_unify_all"
exp_name = "exp_dbg"

net = gitnet

utils.torch_fix_seed(42)
network = net(num_classes=train_ds.fetch_classes(), nb_fils=4, ee_groups=64)
loss_func = FixedLoss(fixed_value=1.0)
# loss_func = torch.nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(network.parameters(), lr=max_lr)
scheduler_t = (torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs, eta_min=0, last_epoch=-1), "epoch")
# device = torch.device("cpu")
device = torch.device("cuda:0") if torch.cuda.is_available() else torch.device("cpu")

ee_trainer = Trainer(network, loss_func, optimizer, scheduler_t, device)

# for e in range(epochs):
#     train_loss, train_acc = ee_trainer.train_1epoch(train_dl)
#     met_dict = {"epoch": e + 1, "train_loss": train_loss, "train_acc": train_acc}
#     ee_trainer.printmet(met_dict, e + 1, epochs, itv=epochs/5)

utils.torch_fix_seed(42)
trainers = []
for i in range(64):
    network = net(num_classes=train_ds.fetch_classes(), nb_fils=4)
    loss_func = FixedLoss(fixed_value=1.0)
    optimizer = torch.optim.SGD(network.parameters(), lr=max_lr)
    scheduler_t = (torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs, eta_min=0, last_epoch=-1), "epoch")
    device = torch.device("cuda:0") if torch.cuda.is_available() else torch.device("cpu")
    # device = torch.device("cpu")

    trainer = Trainer(network, loss_func, optimizer, scheduler_t, device)
    trainers.append(trainer)


In [2]:
network_ee = ee_trainer.network
networks_me = [trainers[i].network for i in range(64)]
ee_sd = network_ee.state_dict()
me_sd = networks_me[0].state_dict()

new_sd = {}
for key, param in ee_sd.items():
    if param.dim() == 0:
        new_sd[key] = param
    else:
        me_tensors = [m.state_dict()[key] for m in networks_me]
        new_sd[key] = torch.cat(me_tensors, dim=0)

network_ee.load_state_dict(new_sd)
# for network_me in networks_me:
    # network_me.load_state_dict(network_me.state_dict())

<All keys matched successfully>

In [3]:
ee_sd = network_ee.state_dict()
sum = 0
for k in me_sd.keys():
    if ee_sd[k].dim() == 0:
        pass
    else:
        channel = ee_sd[k].shape[0] // 64
        # val = (ee_sd[k][:channel] - me_sd[k]).abs().sum().item()
        # sum += val
        # print(k, ee_sd[k].shape, me_sd[k].shape)
        print(ee_sd[k][:channel].var() - me_sd[k].var())
        # print(ee_sd[k][:channel].var(), me_sd[k].var())
# print(sum)
        

tensor(0., device='cuda:0')
tensor(0., device='cuda:0')
tensor(0., device='cuda:0')
tensor(0., device='cuda:0')
tensor(0., device='cuda:0')
tensor(0., device='cuda:0')
tensor(0., device='cuda:0')
tensor(0., device='cuda:0')
tensor(0., device='cuda:0')
tensor(0., device='cuda:0')
tensor(0., device='cuda:0')
tensor(0., device='cuda:0')
tensor(0., device='cuda:0')
tensor(0., device='cuda:0')
tensor(0., device='cuda:0')
tensor(0., device='cuda:0')
tensor(0., device='cuda:0')
tensor(0., device='cuda:0')
tensor(0., device='cuda:0')
tensor(0., device='cuda:0')
tensor(0., device='cuda:0')
tensor(0., device='cuda:0')
tensor(0., device='cuda:0')
tensor(0., device='cuda:0')
tensor(0., device='cuda:0')
tensor(0., device='cuda:0')
tensor(0., device='cuda:0')
tensor(0., device='cuda:0')
tensor(0., device='cuda:0')
tensor(0., device='cuda:0')
tensor(0., device='cuda:0')
tensor(0., device='cuda:0')
tensor(0., device='cuda:0')
tensor(0., device='cuda:0')
tensor(0., device='cuda:0')
tensor(0., device='c

In [4]:
loss_func = torch.nn.CrossEntropyLoss()
loss_func = FixedLoss(fixed_value=1.0)
optimizer = torch.optim.SGD(network.parameters(), lr=max_lr)
scheduler_t = (torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs, eta_min=0, last_epoch=-1), "epoch")
# device = torch.device("cpu")
device = torch.device("cuda:0") if torch.cuda.is_available() else torch.device("cpu")

ee_trainer = Trainer(network_ee, loss_func, optimizer, scheduler_t, device)


loss_func = torch.nn.CrossEntropyLoss()
# loss_func = FixedLoss(fixed_value=1.0)
optimizer = torch.optim.SGD([p for trainer in trainers for p in trainer.network.parameters()], lr=max_lr)
# etrainer.optimizer = torch.optim.Adam([p for trainer in etrainer.me_trainers for p in trainer.network.parameters()], lr=max_lr)
scheduler_t = (torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs, eta_min=0, last_epoch=-1), "epoch")
merge_ens = MergeEnsemble(trainers, loss_func, optimizer, scheduler_t, device)


In [5]:
utils.torch_fix_seed(42)
for e in range(epochs):
    with HookManager(ee_trainer.network) as hm:
        hm.register_forward(module=ee_trainer.network.fc[2], name='layer', fn=lambda module, input, output: output.detach().cpu())
        # hm.register_backward(module=ee_trainer.network.layer1, name='layer', fn=lambda module, input, output: output[0].detach().cpu())
        # train_loss, train_acc = ee_trainer.val_1epoch(train_dl) #----------------------------
        train_loss, train_acc = ee_trainer.train_1epoch(train_dl)
        tsr_ee = hm.get('layer')
    # val_loss, val_acc = ee_trainer.val_1epoch(val_dl)
    met_dict = {"epoch": e + 1, "train_loss": train_loss, "train_acc": train_acc}
    ee_trainer.printmet(met_dict, e + 1, epochs, itv=epochs/5)

    with HookManager(ee_trainer.network) as hm:
        hm.register_forward(module=merge_ens.trainers[0].network.fc[2], name='layer', fn=lambda module, input, output: output.detach().cpu())
        # hm.register_backward(module=merge_ens.trainers[0].network.layer1, name='layer', fn=lambda module, input, output: output[0].detach().cpu())
        # train_loss, train_acc = merge_ens.val_1epoch(train_dl) #----------------------------
        train_loss, train_acc = merge_ens.train_1epoch(train_dl)
        tsr_me = hm.get('layer')
    met_dict = {"epoch": e + 1, "train_loss": train_loss, "train_acc": train_acc}
    merge_ens.printmet(met_dict, e + 1, epochs, itv=epochs/5)

    tsr_ee = torch.cat(tsr_ee)
    tsr_me = torch.cat(tsr_me)
    try:
        channel =  tsr_ee.shape[1] // 64 
    except:
        pass
    a = tsr_ee[:, :channel]
    b = tsr_me

    print(len(tsr_ee), len(tsr_me))
    print(a.shape, b.shape)
    print((a - b).abs().sum().item() / a.numel())




Epoch:   1/   5    train_loss: 1.000000    train_acc: 0.114258    duration: 470.8ms
Epoch:   1/   5    train_loss: 2.303463    train_acc: 0.096680    duration: 3.3s
1024 1024
torch.Size([1024, 10]) torch.Size([1024, 10])
0.17798309326171874
Epoch:   2/   5    train_loss: 1.000000    train_acc: 0.107422    eta: 2025/04/27 20:13 (left: 12.4s)    duration: 3.6s
Epoch:   2/   5    train_loss: 2.302643    train_acc: 0.115234    eta: 2025/04/27 20:13 (left: 12.0s)    duration: 6.3s
1024 1024
torch.Size([1024, 10]) torch.Size([1024, 10])
0.17860777378082277
Epoch:   3/   5    train_loss: 1.000000    train_acc: 0.112305    eta: 2025/04/27 20:13 (left: 9.1s)    duration: 6.6s
Epoch:   3/   5    train_loss: 2.299601    train_acc: 0.126953    eta: 2025/04/27 20:13 (left: 9.1s)    duration: 9.4s
1024 1024
torch.Size([1024, 10]) torch.Size([1024, 10])
0.18194068670272828
Epoch:   4/   5    train_loss: 1.000000    train_acc: 0.109375    eta: 2025/04/27 20:13 (left: 6.1s)    duration: 9.6s
Epoch:   4

In [6]:
tsr_ee = torch.cat(tsr_ee)
tsr_me = torch.cat(tsr_me)
try:
    channel =  tsr_ee.shape[1] // 64 
except:
    pass
a = tsr_ee[:, :channel]
b = tsr_me

print(len(tsr_ee), len(tsr_me))
print(a.shape, b.shape)
print((a - b).abs().sum().item() / a.numel())

TypeError: cat(): argument 'tensors' (position 1) must be tuple of Tensors, not Tensor

In [ ]:
ta = a
tb = b
(ta - tb).abs().sum().item() / ta.numel()

1.2029278195768711e-06

In [ ]:
ta = a[:128]
tb = b[:128]
(ta - tb).abs().sum().item() / ta.numel()

1.2036830412398558e-06

In [ ]:
ta = a[128:256]
tb = b[128:256]
# (ta - tb).abs().sum().item() / ta.numel()


In [ ]:
print(ta[0][0])
print(tb[0][0])

tensor([[0.0000, 2.7058, 0.6128, 0.3102],
        [2.7766, 0.7494, 1.0415, 0.2215],
        [0.5844, 0.0000, 1.7995, 2.7195],
        [0.2120, 1.4647, 2.0220, 0.8864]])
tensor([[0.0000, 2.7058, 0.6128, 0.3102],
        [2.7766, 0.7494, 1.0415, 0.2215],
        [0.5844, 0.0000, 1.7995, 2.7195],
        [0.2120, 1.4647, 2.0220, 0.8864]])


In [ ]:
sd_a = ee_trainer.network.state_dict()
sd_b = me_trainers.trainers[0].network.state_dict()

for key in sd_a.keys():
    tsr_a = sd_a[key].type(torch.float32)
    tsr_b = sd_b[key].type(torch.float32)
    
    try:
        tsr_a = tsr_a[:len(tsr_a) // 64]
    except (IndexError, TypeError):
        pass
    
    # a = tsr_a.var()
    # b = tsr_b.var()
    a = tsr_a.var()
    b = tsr_b.var()
    
    if a > 0.000001 and b > 0.000001:
        print(f"{key:40}: {a:8.6e}, {b:8.6e}, {b/a:8.4f}")

NameError: name 'me_trainers' is not defined